# Petals to the Metal — ResNet50 Inference

Loads trained ResNet50 weights, runs inference on test set,
outputs `submission.csv`. Uses TensorFlow (pre-installed) to read TFRecords.

In [ ]:
import io
from pathlib import Path

import tensorflow as tf
import torch
import torch.nn as nn
import torchvision.models as tv_models
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  PyTorch {torch.__version__}  |  TF {tf.__version__}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# -- Dataset (TensorFlow TFRecord reader) ------------------
class PetalsDataset(Dataset):
    def __init__(self, data_dir, image_size=224, split='train', transform=None):
        if split not in ('train', 'val', 'test'):
            raise ValueError(f"split must be 'train', 'val', or 'test'")
        self.split = split
        self.transform = transform
        self.samples = []

        tfrecord_dir = (Path(data_dir) / f'tfrecords-jpeg-{image_size}x{image_size}' / split)
        tfrecord_paths = sorted(tfrecord_dir.glob('*.tfrec'))
        if not tfrecord_paths:
            raise FileNotFoundError(f"No .tfrec files in '{tfrecord_dir}'")

        feature_desc = (
            {'image': tf.io.FixedLenFeature([], tf.string),
             'class': tf.io.FixedLenFeature([], tf.int64)}
            if split != 'test' else
            {'image': tf.io.FixedLenFeature([], tf.string),
             'id':    tf.io.FixedLenFeature([], tf.string)}
        )

        raw_ds = tf.data.TFRecordDataset([str(p) for p in tfrecord_paths])
        for record in raw_ds:
            parsed = tf.io.parse_single_example(record, feature_desc)
            sample = {'image': parsed['image'].numpy()}
            if split == 'test':
                sample['id'] = parsed['id'].numpy()
            else:
                sample['class'] = parsed['class'].numpy()
            self.samples.append(sample)
        print(f'  {split}: {len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        r = self.samples[idx]
        img = Image.open(io.BytesIO(r['image'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.split == 'test':
            img_id = r['id'].decode('utf-8') if isinstance(r['id'], bytes) else r['id']
            return img, img_id
        return img, int(r['class'])

In [ ]:
# -- Model: ResNet50 (must match models.py structure) -------
class ResNet50(nn.Module):
    def __init__(self, num_classes=104):
        super().__init__()
        self.backbone = tv_models.resnet50(weights=None)
        self.backbone.fc = nn.Linear(2048, num_classes)

    def forward(self, x):
        return self.backbone(x)

model = ResNet50(num_classes=104)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# -- Load trained weights ----------------------------------
# Upload your best_model.pth as a Kaggle Dataset, then set the path below.
MODEL_PATH = '/kaggle/input/YOUR_DATASET_NAME/best_model.pth'  # <-- replace me

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.to(device).eval()
print('Weights loaded.')

In [ ]:
# -- Data --------------------------------------------------
DATA_DIR = '/kaggle/input/tpu-getting-started'

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Loading test data...')
test_ds = PetalsDataset(DATA_DIR, image_size=224, split='test', transform=test_tf)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
print(f'Test images: {len(test_ds)}')

In [ ]:
# -- Inference ---------------------------------------------
ids_all, preds_all = [], []
with torch.no_grad():
    for imgs, img_ids in test_loader:
        logits = model(imgs.to(device))
        preds_all.extend(logits.argmax(dim=1).cpu().tolist())
        ids_all.extend(img_ids)

print(f'Done. {len(ids_all)} predictions.')

with open('submission.csv', 'w') as f:
    f.write('id,label\n')
    for img_id, pred in zip(ids_all, preds_all):
        f.write(f'{img_id},{pred}\n')
print('submission.csv saved')